# ABD-Net + RGA-Net — Palmar (11K Hands)
MBA-Net zaten tamamlandı. Bu notebook sadece ABD-Net ve RGA-Net çalıştırır.

**Çalıştırma öncesi:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Hücre 1 — Kurulum + Drive Bağlantısı
!pip install kagglehub -q

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK"}')

In [ ]:
# Hücre 2 — Dataset İndirme
import os, shutil

# ↓↓↓ TOKEN'INI BURAYA YAPISTIR ↓↓↓
KAGGLE_TOKEN = "KGAT_buraya_kendi_tokenini_yapistir"
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

import kagglehub
print('Dataset indiriliyor...')
cache_path = kagglehub.dataset_download('shyambhu/hands-and-palm-images-dataset')

DATA_DIR   = '/content/data/11k_hands'
IMAGE_ROOT = os.path.join(DATA_DIR, 'Hands', 'Hands')
os.makedirs(IMAGE_ROOT, exist_ok=True)

copied = 0
for root, _, fnames in os.walk(cache_path):
    for fname in fnames:
        src = os.path.join(root, fname)
        dst = os.path.join(
            IMAGE_ROOT if fname.lower().endswith(('.jpg','.jpeg','.png')) else DATA_DIR,
            fname
        )
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
print(f'{copied} dosya kopyalandı | Görüntü: {len(os.listdir(IMAGE_ROOT))}')

In [ ]:
# Hücre 3 — Import'lar ve Sabitler
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import euclidean

HANDINFO_CSV = os.path.join(DATA_DIR, 'HandInfo.csv')
OUTPUT_DIR   = '/content/output'
DRIVE_DIR    = '/content/drive/MyDrive/dl_output'
PLOTS_DIR    = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE       = 224
BATCH_SIZE     = 64
EPOCHS         = 20   # 30'dan 20'ye düşürüldü
LR_INIT        = 1e-4
LR_STEP        = 7
LR_GAMMA       = 0.5
EMBED_DIM      = 512
TRIPLET_MARGIN = 0.3
TRIPLET_WEIGHT = 0.5
MIN_SAMPLES    = 10
TEST_SPLIT     = 0.30
RANDOM_STATE   = 42
FREEZE_EPOCHS  = 5

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print(f'Device: {DEVICE} | Epochs: {EPOCHS}')

In [ ]:
# Hücre 4 — Dataset + DataLoader
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class HandBiometricDataset(Dataset):
    def __init__(self, handinfo_csv, image_root, aspect_filter=None,
                 transform=None, min_samples=MIN_SAMPLES):
        df = pd.read_csv(handinfo_csv)
        df.columns = [c.strip() for c in df.columns]
        img_col = next(c for c in df.columns if 'image' in c.lower() or 'file' in c.lower())
        id_col  = next(c for c in df.columns if 'id' in c.lower() and 'image' not in c.lower())
        asp_col = next(c for c in df.columns if 'aspect' in c.lower())
        if aspect_filter:
            df = df[df[asp_col].str.lower().str.contains(aspect_filter.lower())].copy()
        valid = df[id_col].value_counts()
        df    = df[df[id_col].isin(valid[valid >= min_samples].index)].copy()
        le    = LabelEncoder()
        df['label'] = le.fit_transform(df[id_col].astype(str))
        self.samples       = list(zip(df[img_col].apply(lambda f: os.path.join(image_root, f)), df['label'].tolist()))
        self.transform     = transform
        self.label_encoder = le
        self._labels       = df['label'].to_numpy()
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return (self.transform(img) if self.transform else img), label
    def get_labels(self): return self._labels
    @property
    def num_classes(self): return len(self.label_encoder.classes_)

def make_loaders(aspect_filter='palmar'):
    full_ds = HandBiometricDataset(HANDINFO_CSV, IMAGE_ROOT, aspect_filter)
    labels  = full_ds.get_labels()
    tr_idx, te_idx = train_test_split(np.arange(len(full_ds)), test_size=TEST_SPLIT,
                                       stratify=labels, random_state=RANDOM_STATE)
    tr_ds = HandBiometricDataset(HANDINFO_CSV, IMAGE_ROOT, aspect_filter, TRAIN_TRANSFORM)
    te_ds = HandBiometricDataset(HANDINFO_CSV, IMAGE_ROOT, aspect_filter, TEST_TRANSFORM)
    tr_loader = DataLoader(Subset(tr_ds, tr_idx), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    te_loader = DataLoader(Subset(te_ds, te_idx), BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f'Aspect: {aspect_filter} | Sınıf: {full_ds.num_classes} | Eğitim: {len(tr_idx)} | Test: {len(te_idx)}')
    return tr_loader, te_loader, full_ds.num_classes

print('Dataset sınıfı hazır.')

In [ ]:
# Hücre 5 — Ortak Bloklar
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels//reduction, in_channels, bias=False)
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        att = torch.sigmoid(self.mlp(self.avg_pool(x).view(b,c)) +
                            self.mlp(self.max_pool(x).view(b,c))).view(b,c,1,1)
        return x * att

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        att = torch.sigmoid(self.conv(torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True).values], 1)))
        return x * att

class TripletLoss(nn.Module):
    def __init__(self, margin=TRIPLET_MARGIN):
        super().__init__()
        self.margin = margin
    def forward(self, embeddings, labels):
        diff = embeddings.unsqueeze(0) - embeddings.unsqueeze(1)
        dist = (diff**2).sum(2)
        lbl  = labels.unsqueeze(1)
        same = (lbl == lbl.T).float()
        eye  = torch.eye(dist.size(0), device=dist.device)
        same = same - eye
        pos_dist = (dist * same).max(1).values
        big = dist.max().item() + 1
        mask = (dist > pos_dist.unsqueeze(1)).float() * (1 - same - eye)
        neg_dist = torch.where(mask.bool(), dist, torch.full_like(dist, big)).min(1).values
        return F.relu(pos_dist - neg_dist + self.margin).mean()

print('Ortak bloklar hazır.')

In [ ]:
# Hücre 6 — ABD-Net
class ABDNet(nn.Module):
    def __init__(self, num_classes, embed_dim=EMBED_DIM, lambda_div=1e-3):
        super().__init__()
        self.lambda_div = lambda_div
        bb = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.backbone   = nn.Sequential(*list(bb.children())[:-2])
        self.ch_att     = ChannelAttention(2048)
        self.sp_att     = SpatialAttention()
        self.att_pool   = nn.AdaptiveAvgPool2d(1)
        self.att_bn     = nn.BatchNorm1d(2048)
        self.att_fc     = nn.Linear(2048, embed_dim)
        self.base_pool  = nn.AdaptiveAvgPool2d(1)
        self.base_bn    = nn.BatchNorm1d(2048)
        self.base_fc    = nn.Linear(2048, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        feat = self.backbone(x)
        a = self.att_fc(self.att_bn(self.att_pool(self.sp_att(self.ch_att(feat))).flatten(1)))
        b = self.base_fc(self.base_bn(self.base_pool(feat).flatten(1)))
        div_loss = (torch.mm(self.att_fc.weight, self.base_fc.weight.T) -
                    torch.eye(self.att_fc.weight.size(0), device=self.att_fc.weight.device)).pow(2).sum()
        emb = F.normalize(a, p=2, dim=1)
        return self.classifier(emb), emb, div_loss

print('ABD-Net hazır.')

In [ ]:
# Hücre 7 — RGA-Net
class RGAModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.merge = nn.Conv2d(2*in_channels, in_channels, 1, bias=False)
        self.bn    = nn.BatchNorm2d(in_channels)
    def forward(self, x):
        B, C, H, W = x.shape
        f = x.view(B, C, H*W)
        R = F.softmax(torch.bmm(f.permute(0,2,1), f), dim=2)
        rel = torch.bmm(f, R.permute(0,2,1)).view(B, C, H, W)
        return x * torch.sigmoid(self.bn(self.merge(torch.cat([x, rel], 1))))

class RGANet(nn.Module):
    def __init__(self, num_classes, embed_dim=EMBED_DIM):
        super().__init__()
        bb = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.layer0 = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.rga3   = RGAModule(1024)
        self.rga4   = RGAModule(2048)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.bn     = nn.BatchNorm1d(2048)
        self.drop   = nn.Dropout(0.4)
        self.fc     = nn.Linear(2048, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.rga3(self.layer3(x))
        x = self.rga4(self.layer4(x))
        emb = F.normalize(self.fc(self.drop(self.bn(self.pool(x).flatten(1)))), p=2, dim=1)
        return self.classifier(emb), emb

print('RGA-Net hazır.')

In [ ]:
# Hücre 8 — Eğitim + Değerlendirme
def train_one_epoch(model, loader, optimizer, ce, triplet, is_abd):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if is_abd:
            logits, embs, div = model(imgs)
            loss = ce(logits, labels) + TRIPLET_WEIGHT*triplet(embs, labels) + model.lambda_div*div
        else:
            logits, embs = model(imgs)
            loss = ce(logits, labels) + TRIPLET_WEIGHT*triplet(embs, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item()*imgs.size(0)
        correct += (logits.argmax(1)==labels).sum().item()
        total   += imgs.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, is_abd):
    model.eval()
    embs_list, lbl_list, pred_list = [], [], []
    for imgs, labels in loader:
        out = model(imgs.to(DEVICE))
        logits, embs = (out[0], out[1])
        embs_list.append(embs.cpu().numpy())
        lbl_list.append(labels.numpy())
        pred_list.append(logits.argmax(1).cpu().numpy())
    embs = np.concatenate(embs_list)
    lbls = np.concatenate(lbl_list)
    acc  = (np.concatenate(pred_list)==lbls).mean()
    return acc, embs, lbls

def train_model(model, tr_loader, te_loader, model_name):
    is_abd = model_name == 'ABD-Net'
    backbone_params = ['backbone','layer0','layer1','layer2','layer3','layer4','rga3','rga4','ch_att','sp_att']
    def set_bb(req):
        for n, p in model.named_parameters():
            if any(n.startswith(k) for k in backbone_params): p.requires_grad = req
    set_bb(False)
    opt  = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_INIT, weight_decay=5e-4)
    sch  = torch.optim.lr_scheduler.StepLR(opt, LR_STEP, LR_GAMMA)
    ce   = nn.CrossEntropyLoss()
    trip = TripletLoss()
    hist = {'train_loss':[], 'train_acc':[], 'test_acc':[]}
    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            set_bb(True)
            opt.add_param_group({'params': [p for n,p in model.named_parameters()
                                            if any(n.startswith(k) for k in backbone_params)],
                                 'lr': LR_INIT*0.1})
            print(f'  → Epoch {epoch+1}: backbone açıldı')
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, opt, ce, trip, is_abd)
        te_acc, _, _    = evaluate(model, te_loader, is_abd)
        sch.step()
        hist['train_loss'].append(tr_loss)
        hist['train_acc'].append(tr_acc)
        hist['test_acc'].append(te_acc)
        print(f'  Epoch {epoch+1:02d}/{EPOCHS} | loss={tr_loss:.4f} | train={tr_acc*100:.2f}% | test={te_acc*100:.2f}%')
    _, te_embs, te_lbls = evaluate(model, te_loader, is_abd)
    _, tr_embs, tr_lbls = evaluate(model, tr_loader, is_abd)
    return model, hist, tr_embs, tr_lbls, te_embs, te_lbls

print('Eğitim fonksiyonları hazır.')

In [ ]:
# Hücre 9 — Metrik + Grafik Fonksiyonları
def compute_cmc(tr_embs, tr_lbls, te_embs, te_lbls, max_rank=10):
    from sklearn.metrics.pairwise import euclidean_distances
    dist = euclidean_distances(te_embs, tr_embs)
    cmc  = np.zeros(max_rank)
    for i, lbl in enumerate(te_lbls):
        sl = tr_lbls[np.argsort(dist[i])]
        for r in range(max_rank):
            if sl[r] == lbl: cmc[r:] += 1; break
    return cmc / len(te_lbls)

def compute_eer(tr_embs, tr_lbls, te_embs, te_lbls, n=500):
    classes   = np.unique(tr_lbls)
    centroids = {c: tr_embs[tr_lbls==c].mean(0) for c in classes}
    gen, imp  = [], []
    rng = np.random.default_rng(RANDOM_STATE)
    for i in range(len(te_embs)):
        lbl = te_lbls[i]
        if lbl not in centroids: continue
        gen.append(-euclidean(te_embs[i], centroids[lbl]))
        for c in rng.choice([x for x in classes if x!=lbl], size=min(10,len(classes)-1), replace=False):
            imp.append(-euclidean(te_embs[i], centroids[c]))
    gen, imp = np.array(gen), np.array(imp)
    thr = np.linspace(np.concatenate([gen,imp]).min(), np.concatenate([gen,imp]).max(), n)
    far = np.array([np.mean(imp>=t) for t in thr])
    frr = np.array([np.mean(gen< t) for t in thr])
    idx = np.argmin(np.abs(far-frr))
    return thr, far, frr, (far[idx]+frr[idx])/2, thr[idx]

def save_plots(hist, thr, far, frr, eer, eer_thr, tag):
    # Eğitim eğrisi
    fig, (a1,a2) = plt.subplots(1,2,figsize=(13,5))
    a1.plot(hist['train_loss'], color='steelblue', lw=2)
    a1.set_title(f'Kayıp — {tag}'); a1.set_xlabel('Epoch'); a1.grid(alpha=0.4)
    a2.plot(hist['train_acc'], label='Eğitim', color='steelblue', lw=2)
    a2.plot(hist['test_acc'],  label='Test',   color='coral',     lw=2)
    a2.set_title(f'Doğruluk — {tag}'); a2.legend(); a2.grid(alpha=0.4)
    plt.tight_layout()
    for d in [PLOTS_DIR, os.path.join(DRIVE_DIR,'plots')]:
        os.makedirs(d, exist_ok=True)
        plt.savefig(os.path.join(d, f'training_{tag}.png'), dpi=150)
    plt.show(); plt.close()
    # FAR/FRR
    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot(thr, far, color='red',  lw=2, label='FAR')
    ax.plot(thr, frr, color='blue', lw=2, label='FRR')
    ax.axvline(eer_thr, color='green', ls='--', lw=1.5, label=f'EER={eer*100:.2f}%')
    ax.set_title(f'FAR/FRR — {tag}'); ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout()
    for d in [PLOTS_DIR, os.path.join(DRIVE_DIR,'plots')]:
        plt.savefig(os.path.join(d, f'far_frr_{tag}.png'), dpi=150)
    plt.show(); plt.close()

print('Metrik ve grafik fonksiyonları hazır.')

In [ ]:
# Hücre 10 — ABD-Net Çalıştır
print('\n' + '='*60)
print('  DENEY: ABD-Net_palmar')
print('='*60)

tr_loader, te_loader, num_classes = make_loaders('palmar')
abd_model = ABDNet(num_classes=num_classes).to(DEVICE)
print(f'Parametre sayısı: {sum(p.numel() for p in abd_model.parameters()):,}')

abd_model, abd_hist, abd_tr_embs, abd_tr_lbls, abd_te_embs, abd_te_lbls = \
    train_model(abd_model, tr_loader, te_loader, 'ABD-Net')

abd_cmc = compute_cmc(abd_tr_embs, abd_tr_lbls, abd_te_embs, abd_te_lbls)
abd_thr, abd_far, abd_frr, abd_eer, abd_eer_thr = compute_eer(abd_tr_embs, abd_tr_lbls, abd_te_embs, abd_te_lbls)

print(f'\n  Sonuçlar:')
print(f'    Rank-1  : {abd_cmc[0]*100:.2f}%')
print(f'    Accuracy: {abd_hist["test_acc"][-1]*100:.2f}%')
print(f'    EER     : {abd_eer*100:.2f}%')

save_plots(abd_hist, abd_thr, abd_far, abd_frr, abd_eer, abd_eer_thr, 'ABDNet_palmar')

# Model + sonucu Drive'a kaydet
torch.save(abd_model.state_dict(), os.path.join(DRIVE_DIR, 'ABDNet_palmar_model.pt'))
print('Model Drive\'a kaydedildi.')

In [ ]:
# Hücre 11 — RGA-Net Çalıştır
print('\n' + '='*60)
print('  DENEY: RGA-Net_palmar')
print('='*60)

rga_model = RGANet(num_classes=num_classes).to(DEVICE)
print(f'Parametre sayısı: {sum(p.numel() for p in rga_model.parameters()):,}')

rga_model, rga_hist, rga_tr_embs, rga_tr_lbls, rga_te_embs, rga_te_lbls = \
    train_model(rga_model, tr_loader, te_loader, 'RGA-Net')

rga_cmc = compute_cmc(rga_tr_embs, rga_tr_lbls, rga_te_embs, rga_te_lbls)
rga_thr, rga_far, rga_frr, rga_eer, rga_eer_thr = compute_eer(rga_tr_embs, rga_tr_lbls, rga_te_embs, rga_te_lbls)

print(f'\n  Sonuçlar:')
print(f'    Rank-1  : {rga_cmc[0]*100:.2f}%')
print(f'    Accuracy: {rga_hist["test_acc"][-1]*100:.2f}%')
print(f'    EER     : {rga_eer*100:.2f}%')

save_plots(rga_hist, rga_thr, rga_far, rga_frr, rga_eer, rga_eer_thr, 'RGANet_palmar')

torch.save(rga_model.state_dict(), os.path.join(DRIVE_DIR, 'RGANet_palmar_model.pt'))
print('Model Drive\'a kaydedildi.')

In [ ]:
# Hücre 12 — Sonuç Tablosu + Drive'a Kaydet
results = [
    {'Model':'MBA-Net', 'Rank-1':'99.81%', 'Accuracy':'97.26%', 'EER':'0.13%'},  # önceki run
    {'Model':'ABD-Net', 'Rank-1':f'{abd_cmc[0]*100:.2f}%',
     'Accuracy':f'{abd_hist["test_acc"][-1]*100:.2f}%', 'EER':f'{abd_eer*100:.2f}%'},
    {'Model':'RGA-Net', 'Rank-1':f'{rga_cmc[0]*100:.2f}%',
     'Accuracy':f'{rga_hist["test_acc"][-1]*100:.2f}%', 'EER':f'{rga_eer*100:.2f}%'},
]
df = pd.DataFrame(results)
df.to_csv(os.path.join(DRIVE_DIR, 'dl_results_summary.csv'), index=False)

print('\n' + '='*50)
print('TAM SONUÇ TABLOSU — Palmar')
print('='*50)
display(df)

# Bar grafik
fig, (a1,a2) = plt.subplots(1,2,figsize=(12,5))
colors = ['steelblue','coral','seagreen']
ranks  = [float(r['Rank-1'].strip('%')) for r in results]
eers   = [float(r['EER'].strip('%'))   for r in results]
names  = [r['Model'] for r in results]
a1.bar(names, ranks, color=colors)
a1.set_ylabel('Rank-1 (%)'); a1.set_title('Rank-1 Doğruluğu'); a1.set_ylim(0,105)
[a1.text(i, v+0.5, f'{v:.1f}%', ha='center') for i,v in enumerate(ranks)]
a2.bar(names, eers,  color=colors)
a2.set_ylabel('EER (%)');    a2.set_title('EER Karşılaştırması'); a2.set_ylim(0,40)
[a2.text(i, v+0.2, f'{v:.2f}%', ha='center') for i,v in enumerate(eers)]
plt.tight_layout()
for d in [PLOTS_DIR, os.path.join(DRIVE_DIR,'plots')]:
    os.makedirs(d, exist_ok=True)
    plt.savefig(os.path.join(d,'dl_comparison.png'), dpi=150)
plt.show(); plt.close()
print('Tüm sonuçlar Drive/dl_output/ klasörüne kaydedildi.')